In [75]:
import pandas as pd
import numpy as np

1. Create a pandas series from each of the items below: a list, numpy and a dictionary

In [76]:
mylist = list('abcedfghijklmnopqrstuvwxyz')
myarr = np.arange(26)
mydict = dict(zip(mylist, myarr))

mylist = pd.Series(mylist)
myarr = pd.Series(myarr)
mydict = pd.Series(mydict)


2. Convert the series `ser` into a dataframe with its index as another column on the dataframe.

In [77]:
ser = pd.Series(mydict)
df = ser.to_frame().reset_index()
df.head()


,index,0
0,a,0
1,b,1
2,c,2
3,e,3
4,d,4


3. Combine ser1 and ser2 to form a dataframe.
4. Give a name to the series ser1 calling it ‘alphabets’.

In [78]:
ser1 = pd.Series(list('abcedfghijklmnopqrstuvwxyz'))
ser2 = pd.Series(np.arange(26))
combined = pd.concat([ser1, ser2], axis=1)

ser1.name = "alphabets"


5. From ser1 remove items present in ser2.

In [79]:
ser1 = pd.Series([1, 2, 3, 4, 5])
ser2 = pd.Series([4, 5, 6, 7, 8])

ser1[~ser1.isin(ser2)]

0    1
1    2
2    3
dtype: int64

6. Get all items of ser1 and ser2 not common to both.

In [80]:
ser_u = pd.Series(np.union1d(ser1, ser2))  # union
ser_i = pd.Series(np.intersect1d(ser1, ser2))  # intersect
ser_u[~ser_u.isin(ser_i)]

0    1
1    2
2    3
5    6
6    7
7    8
dtype: int64

7. Compute the minimum, 25th percentile, median, 75th, and maximum of ser.

In [81]:
ser = pd.Series(np.random.normal(10, 5, 25))
np.percentile(ser, q=[0, 25, 50, 75, 100])
# or can use this ser.describe()

array([ 0.6705216 ,  6.21539294,  9.55642044, 11.79287729, 23.27289553])

8. Calculate the frequency counts of each unique value ser.

In [82]:
ser = pd.Series(np.take(list('abcdefgh'), np.random.randint(8, size=30)))
ser.value_counts()

f    6
e    5
b    4
h    4
g    4
d    3
c    3
a    1
Name: count, dtype: int64

9. From ser, keep the top 2 most frequent items as it is and replace everything else as '10000'.

In [83]:
np.random.RandomState(100)
ser = pd.Series(np.random.randint(1, 5, [12]))

print("Top 2 Freq:", ser.value_counts())
ser[~ser.isin(ser.value_counts().index[:2])] = 10000
ser

Top 2 Freq: 1    5
4    4
2    3
Name: count, dtype: int64


0     10000
1         4
2         4
3         4
4     10000
5         1
6         1
7     10000
8         1
9         4
10        1
11        1
dtype: int64

10. Reshape the series ser into a dataframe with 7 rows and 5 columns

In [84]:
ser = pd.Series(np.random.randint(1, 10, 35))
df = pd.DataFrame(ser.values.reshape(7,5))
df


,0,1,2,3,4
0,6,3,8,3,6
1,2,9,6,2,8
2,1,3,7,7,7
3,2,1,9,4,7
4,2,7,4,7,7
5,5,9,7,8,7
6,1,7,7,6,4


**Task of the day**
Practice cleaning a messy dataset

In [85]:
import kagglehub
import pandas as pd
import numpy as np

path = kagglehub.dataset_download("kandeelai22/messy-e-commerce-sales-dataset")
df = pd.read_csv("/Users/liorflaxman/Desktop/AI_form_analysis_project/Datasets/messy_ecommerce_sales_data.csv")

print(f"Loaded Dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded Dataset: 103 rows, 11 columns


In [86]:
df.head(5)

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
0,100,Customer_100,ORD-41285,11/22/2024,Blender,Home,3,38,Cash on Delivery,Shipped,114.00
1,101,Customer_101,ORD-35783,7/5/2025,Smartphone,Electronics,2,abd,PayPal,Processing,NaN
2,102,Customer_102,ORD-84355,12/23/2024,Tennis Racket,Sports,1,389.05,PayPal,Delivered,389.05
3,103,Customer_103,ORD-57811,3/19/2025,Science,Books,5,233.92,PayPal,Processing,1169.60
4,104,Customer_104,ORD-93614,10/20/2025,Biography,Books,1,552.51,Cash on Delivery,Processing,552.51


# Issues to fix in datset
1. price has some str values and float values
2. total has null values and minus values
3. order_date is not all in same format
4. quantity has minus numbers and Nan

In [87]:
#1. HEADER CLEANING
print(f"Messy columns{df.columns.tolist()}")
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print(f"New Columns {df.columns.tolist()}")


Messy columns['ID', ' Customer_Name', 'Order_ID', 'Order_Date', 'Product', ' Category', 'Quantity', 'Price', 'Payment_Method', 'Status', 'Total']
New Columns ['id', 'customer_name', 'order_id', 'order_date', 'product', 'category', 'quantity', 'price', 'payment_method', 'status', 'total']


In [88]:
#2. TYPE CONVERSION & CURRENCY CLEANING

dirty_price = df['price'].astype(str).str.isalnum()
print(df.loc[dirty_price, ['id', 'price']].head(10))

df['price'] = df['price'].astype(str).str.replace(r'[^\d.]', '', regex=True)
df['price'] = pd.to_numeric(df['price']).round(2)
print(df.loc[dirty_price, ['id', 'price']].head(10))


     id  price
0   100     38
1   101    abd
17  117  10000
96  196    abd
     id    price
0   100     38.0
1   101      NaN
17  117  10000.0
96  196      NaN


In [89]:
dirty_total = df['total'].astype(str).str.isalnum()
print(df.loc[dirty_total, ['id', 'total']].head(10))

df['total'] = df['total'].astype(str).str.replace(r'[^\d.]', '', regex=True)
df['total'] = pd.to_numeric(df['total']).round(2)
print(df.loc[dirty_total, ['id', 'total']].head(10))

Empty DataFrame
Columns: [id, total]
Index: []
Empty DataFrame
Columns: [id, total]
Index: []


In [90]:
dirty_quantity = df['quantity'].astype(str).str.isalnum()
print(df.loc[dirty_quantity, ['id', 'quantity']].head(10))

df['quantity'] = df['quantity'].astype(str).str.replace(r'[^\d.]', '', regex=True)
df['quantity'] = pd.to_numeric(df['quantity']).round(2)
print(df.loc[dirty_quantity, ['id', 'quantity']].head(10))

     id quantity
0   100        3
1   101        2
2   102        1
3   103        5
4   104        1
5   105        3
7   107        5
8   108        1
9   109        5
10  110        5
     id  quantity
0   100       3.0
1   101       2.0
2   102       1.0
3   103       5.0
4   104       1.0
5   105       3.0
7   107       5.0
8   108       1.0
9   109       5.0
10  110       5.0


In [91]:
#3. CATEGORICAL TYPOS
print(f"Messy Categories: {df['category'].unique()}")

clean_up_map = {
    'electronic': 'Electronics',
    'ELECTRONICS': 'Electronics', 
    'electronics': 'Electronics',
    'sports': 'Sports',
    'N/A': np.nan
}

df['category'] = df['category'].replace(clean_up_map)
print(f"Cleaned Categories: {df['category'].unique()}")


Messy Categories: <StringArray>
[       'Home', 'Electronics',      'Sports',       'Books',      'sports',
    'Clothing',  'electronic', 'ELECTRONICS', 'electronics',           nan]
Length: 10, dtype: str
Cleaned Categories: <StringArray>
['Home', 'Electronics', 'Sports', 'Books', 'Clothing', nan]
Length: 6, dtype: str


In [92]:
#4. DATE PARSING
print(df['order_date'].dtype)

df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
print(df['order_date'].dtype)


str
datetime64[us]


In [93]:
#6. MAKING SURE TOTAL IS CALCULATED CORRECTLY
incorrect_total = df['total'] != df['quantity'] * df['price']
print(df.loc[incorrect_total, ['quantity', 'price', 'total']])

df['total'] = df['quantity'] * df['price']
print(df.loc[incorrect_total, ['quantity', 'price', 'total']])



     quantity   price    total
1         2.0     NaN      NaN
6         NaN  978.63      NaN
7         5.0  587.68  2938.40
10        5.0     NaN      NaN
15        2.0  796.84  1115.58
16        4.0     NaN      NaN
20        5.0  300.00      NaN
24        5.0     NaN      NaN
30        2.0     NaN      NaN
34        5.0  591.53  2957.65
42        5.0  645.26  2258.41
51        2.0  457.16   640.02
56        2.0     NaN      NaN
60        5.0  242.52  1212.60
64        NaN  587.64      NaN
67        NaN  593.93      NaN
73        5.0  351.89  1759.45
79        3.0  868.67  2606.01
83        5.0     NaN      NaN
85        2.0  730.92  1023.29
92        4.0  203.63      NaN
93        NaN  522.02      NaN
96        NaN     NaN      NaN
99        5.0  372.28  1861.40
100       1.0  111.36    77.95
     quantity   price    total
1         2.0     NaN      NaN
6         NaN  978.63      NaN
7         5.0  587.68  2938.40
10        5.0     NaN      NaN
15        2.0  796.84  1593.68
16      

In [94]:
df.head(10)

,id,customer_name,order_id,order_date,product,category,quantity,price,payment_method,status,total
0,100,Customer_100,ORD-41285,2024-11-22,Blender,Home,3.0,38.00,Cash on Delivery,Shipped,114.00
1,101,Customer_101,ORD-35783,2025-07-05,Smartphone,Electronics,2.0,NaN,PayPal,Processing,NaN
2,102,Customer_102,ORD-84355,2024-12-23,Tennis Racket,Sports,1.0,389.05,PayPal,Delivered,389.05
3,103,Customer_103,ORD-57811,2025-03-19,Science,Books,5.0,233.92,PayPal,Processing,1169.60
4,104,Customer_104,ORD-93614,2025-10-20,Biography,Books,1.0,552.51,Cash on Delivery,Processing,552.51
5,105,Customer_105,ORD-22442,2024-11-20,Tennis Racket,Sports,3.0,122.06,Cash on Delivery,Cancelled,366.18
6,106,Customer_106,ORD-25885,2025-02-02,Blender,Home,NaN,978.63,Bank Transfer,Processing,NaN
7,107,Customer_107,ORD-89122,2025-01-03,Biography,Books,5.0,587.68,PayPal,Returned,2938.40
8,108,Customer_108,ORD-64400,2025-10-23,Science,Books,1.0,600.29,Cash on Delivery,Processing,600.29
9,109,Customer_109,ORD-18512,2025-05-03,Tennis Racket,Sports,5.0,168.34,Credit Card,Shipped,841.70


In [95]:
display_nan_rows = df[df.isna().any(axis=1)]
print(display_nan_rows)

df['price'] = df['price'].fillna(df.groupby('product')['price'].transform('mean'))
df['price'] = pd.to_numeric(df['price'], errors='coerce').round(2)
print(display_nan_rows)


     id customer_name   order_id order_date        product     category  \
1   101  Customer_101  ORD-35783 2025-07-05     Smartphone  Electronics   
6   106  Customer_106  ORD-25885 2025-02-02        Blender         Home   
10  110  Customer_110  ORD-61020 2025-09-26     Basketball       Sports   
14  114  Customer_114  ORD-77417        NaT     Smartphone  Electronics   
16  116  Customer_116  ORD-63660 2025-10-30      Microwave         Home   
24  124  Customer_124  ORD-46136 2025-05-31         Comics        Books   
30  130  Customer_130  ORD-34007 2025-08-15  Tennis Racket       Sports   
33  133  Customer_133  ORD-68182 2024-12-05      Biography          NaN   
36  136  Customer_136  ORD-20985 2025-06-12     Headphones          NaN   
56  156  Customer_156  ORD-34679 2024-11-28          Jeans     Clothing   
64  164  Customer_164  ORD-23010 2025-02-09  Tennis Racket       Sports   
67  167  Customer_167  ORD-30329 2025-08-07     Basketball       Sports   
80  180  Customer_180  OR

In [96]:
display_nan_rows


,id,customer_name,order_id,order_date,product,category,quantity,price,payment_method,status,total
1,101,Customer_101,ORD-35783,2025-07-05,Smartphone,Electronics,2.0,NaN,PayPal,Processing,NaN
6,106,Customer_106,ORD-25885,2025-02-02,Blender,Home,NaN,978.63,Bank Transfer,Processing,NaN
10,110,Customer_110,ORD-61020,2025-09-26,Basketball,Sports,5.0,NaN,Cash on Delivery,Cancelled,NaN
14,114,Customer_114,ORD-77417,NaT,Smartphone,Electronics,2.0,413.13,Cash on Delivery,Shipped,826.26
16,116,Customer_116,ORD-63660,2025-10-30,Microwave,Home,4.0,NaN,Cash on Delivery,Processing,NaN
24,124,Customer_124,ORD-46136,2025-05-31,Comics,Books,5.0,NaN,PayPal,Cancelled,NaN
30,130,Customer_130,ORD-34007,2025-08-15,Tennis Racket,Sports,2.0,NaN,Bank Transfer,Processing,NaN
33,133,Customer_133,ORD-68182,2024-12-05,Biography,NaN,5.0,343.24,Credit Card,Shipped,1716.20
36,136,Customer_136,ORD-20985,2025-06-12,Headphones,NaN,1.0,696.71,Credit Card,Delivered,696.71
56,156,Customer_156,ORD-34679,2024-11-28,Jeans,Clothing,2.0,NaN,Bank Transfer,Shipped,NaN


In [104]:
print(df['product'].unique())

category_map = {
    'Biography': 'Books',
    'Headphones': 'Electronics',
    'Basketball': 'Sports',
    'Vacuum': 'Home',
    'Jeans': 'Clothing',
    'Shoes': 'Clothing',
    'Smartphone': 'Electronics',
    'Laptop': 'Electronics'
}
df['category'] = df['category'].fillna(df['product'].map(category_map))
#'Home', 'Electronics', 'Sports', 'Books', 'Clothing',
df.head(10)


<StringArray>
[      'Blender',    'Smartphone', 'Tennis Racket',       'Science',
     'Biography',    'Basketball',        'Comics',         'Shoes',
       'Fiction',     'Microwave',      'Yoga Mat',        'Vacuum',
          'Lamp',       'T-shirt',    'Smartwatch',    'Headphones',
      'Football',        'Laptop',        'Jacket',         'shoes',
         'Jeans']
Length: 21, dtype: str


,id,customer_name,order_id,order_date,product,category,quantity,price,payment_method,status,total
0,100,Customer_100,ORD-41285,2024-11-22,Blender,Home,3.0,38.00,Cash on Delivery,Shipped,114.00
1,101,Customer_101,ORD-35783,2025-07-05,Smartphone,Electronics,2.0,560.99,PayPal,Processing,NaN
2,102,Customer_102,ORD-84355,2024-12-23,Tennis Racket,Sports,1.0,389.05,PayPal,Delivered,389.05
3,103,Customer_103,ORD-57811,2025-03-19,Science,Books,5.0,233.92,PayPal,Processing,1169.60
4,104,Customer_104,ORD-93614,2025-10-20,Biography,Books,1.0,552.51,Cash on Delivery,Processing,552.51
5,105,Customer_105,ORD-22442,2024-11-20,Tennis Racket,Sports,3.0,122.06,Cash on Delivery,Cancelled,366.18
6,106,Customer_106,ORD-25885,2025-02-02,Blender,Home,NaN,978.63,Bank Transfer,Processing,NaN
7,107,Customer_107,ORD-89122,2025-01-03,Biography,Books,5.0,587.68,PayPal,Returned,2938.40
8,108,Customer_108,ORD-64400,2025-10-23,Science,Books,1.0,600.29,Cash on Delivery,Processing,600.29
9,109,Customer_109,ORD-18512,2025-05-03,Tennis Racket,Sports,5.0,168.34,Credit Card,Shipped,841.70
